# Replication: Hua, Cheng & Wang (2010), Table 1

**Paper:** Hua, G., Cheng, T.C.E., & Wang, S. (2010). *Managing Carbon Footprints in Inventory Control.* SSRN: https://ssrn.com/abstract=1628953

This notebook reproduces the seven numerical examples printed in Table 1 (page 16). Each row computes, for given parameters $(K, h, e, g, C, \alpha)$ and demand $D = 60{,}000$:

- $\hat{Q}$ — order size that minimizes carbon footprint
- $Q^*$ — optimal order size under cap-and-trade
- $Q^0$ — classical EOQ order size
- $\alpha_0$ — buy/sell threshold (Theorem 3)
- $X$ — transfer quantity (positive = sell credit, negative = buy)
- $TC(Q^*)$ — total cost
- $\Delta CO_2$ — carbon savings vs. EOQ baseline
- $\Delta TC$ — cost change vs. EOQ baseline

Figures 2–6 and the Theorem-1-to-5 verification are reproduced in follow-up notebooks.

## Model equations

From Section 2.1 of the paper (simplified case $e_0 = g_0 = 0$):

$$Q^0 = \sqrt{\frac{2KD}{h}}, \qquad \hat{Q} = \sqrt{\frac{2eD}{g}}, \qquad Q^* = \sqrt{\frac{2(K+Ce)D}{h + Cg}}$$

$$TC_0 = \sqrt{2KDh}, \qquad TC(Q^*) = \sqrt{2D(K+Ce)(h+Cg)} \; - \; C\alpha$$

$$\alpha_0 = e\sqrt{\frac{(h+Cg)D}{2(K+Ce)}} \; + \; g\sqrt{\frac{(K+Ce)D}{2(h+Cg)}}$$

$$CF(Q) = \frac{eD}{Q} + \frac{gQ}{2}, \qquad X = \alpha - CF(Q^*)$$

These are implemented in `model.py` and imported below.

In [1]:
import numpy as np
import pandas as pd

from model import (
    Q_eoq, Q_hat, Q_star,
    TC_eoq, TC_star,
    alpha_threshold, transfer,
    delta_CO2, delta_TC,
)

## Table 1 inputs

Global parameter: demand $D = 60{,}000$. Six rows use $C = 0.2$ and $\alpha = 8{,}000$; Row 4 uses $C = 0.3$ and $\alpha = 10{,}000$ (paper footnote on Table 1, page 16).

In [2]:
D = 60_000

# (label, K, h, e, g, C, alpha)
cases = [
    ("Row 1",  180, 0.30, 600, 1.0, 0.2,  8_000),
    ("Row 2",  200, 0.40, 500, 1.0, 0.2,  8_000),
    ("Row 3",  200, 0.36, 450, 1.0, 0.2,  8_000),
    ("Row 4*", 250, 0.40, 540, 1.5, 0.3, 10_000),
    ("Row 5",  250, 0.40, 540, 1.5, 0.2,  8_000),
    ("Row 6",  200, 0.36, 800, 1.0, 0.2,  8_000),
    ("Row 7",  250, 0.45, 900, 1.0, 0.2,  8_000),
]

pd.DataFrame(cases, columns=["case", "K", "h", "e", "g", "C", "alpha"])

,case,K,h,e,g,C,alpha
0,Row 1,180,0.30,600,1.0,0.2,8000
1,Row 2,200,0.40,500,1.0,0.2,8000
2,Row 3,200,0.36,450,1.0,0.2,8000
3,Row 4*,250,0.40,540,1.5,0.3,10000
4,Row 5,250,0.40,540,1.5,0.2,8000
5,Row 6,200,0.36,800,1.0,0.2,8000
6,Row 7,250,0.45,900,1.0,0.2,8000


## Computed values

In [3]:
records = []
for label, K, h, e, g, C, alpha in cases:
    records.append({
        "case":    label,
        "Q_hat":   round(Q_hat(e, g, D)),
        "Q_star":  round(Q_star(K, h, e, g, C, D)),
        "Q_0":     round(Q_eoq(K, h, D)),
        "alpha_0": round(alpha_threshold(K, h, e, g, C, D)),
        "X":       round(transfer(K, h, e, g, C, alpha, D)),
        "TC_star": round(TC_star(K, h, e, g, C, alpha, D)),
        "dCO2":    round(delta_CO2(K, h, e, g, C, D)),
        "dTC":     round(delta_TC(K, h, e, g, C, alpha, D)),
    })

computed = pd.DataFrame(records)
computed

,case,Q_hat,Q_star,Q_0,alpha_0,X,TC_star,dCO2,dTC
0,Row 1,8485,8485,8485,8485,-485,2643,0,97
1,Row 2,7746,7746,7746,7746,254,3048,0,-51
2,Row 3,7348,7883,8165,7367,633,2815,23,-125
3,Row 4*,6573,7627,8660,9968,32,3483,268,18
4,Row 5,6573,7834,8660,10011,-2011,3884,225,420
5,Row 6,9798,8783,8165,9857,-1857,3319,105,379
6,Row 7,10392,8910,8165,10516,-2516,4191,180,517


## Paper's reported values

Transcribed verbatim from Table 1 (page 16) of the paper.

In [4]:
paper = pd.DataFrame([
    ("Row 1",   8453,  8453,  8453,  8485,  -485, 2643,    0,   97),
    ("Row 2",   7746,  7746,  7746,  7746,   254, 3048,    0,  -51),
    ("Row 3",   7348,  7883,  8165,  7367,   633, 2815,   23, -125),
    ("Row 4*",  6573,  7627,  8660,  9968,    32, 3483,  268,   18),
    ("Row 5",   6573,  7834,  8660, 10011, -2011, 3884,  225,  420),
    ("Row 6",   9798,  8783,  8165,  9857, -1857, 3319,  105,  379),
    ("Row 7",  10392,  8910,  8165, 10516, -2516, 4191,  181,  517),
], columns=["case", "Q_hat", "Q_star", "Q_0", "alpha_0", "X", "TC_star", "dCO2", "dTC"])

paper

,case,Q_hat,Q_star,Q_0,alpha_0,X,TC_star,dCO2,dTC
0,Row 1,8453,8453,8453,8485,-485,2643,0,97
1,Row 2,7746,7746,7746,7746,254,3048,0,-51
2,Row 3,7348,7883,8165,7367,633,2815,23,-125
3,Row 4*,6573,7627,8660,9968,32,3483,268,18
4,Row 5,6573,7834,8660,10011,-2011,3884,225,420
5,Row 6,9798,8783,8165,9857,-1857,3319,105,379
6,Row 7,10392,8910,8165,10516,-2516,4191,181,517


## Difference (replicated − paper)

In [5]:
metrics = ["Q_hat", "Q_star", "Q_0", "alpha_0", "X", "TC_star", "dCO2", "dTC"]
diff = computed.set_index("case")[metrics] - paper.set_index("case")[metrics]
diff

,Q_hat,Q_star,Q_0,alpha_0,X,TC_star,dCO2,dTC
case,,,,,,,,
Row 1,32,32,32,0,0,0,0,0
Row 2,0,0,0,0,0,0,0,0
Row 3,0,0,0,0,0,0,0,0
Row 4*,0,0,0,0,0,0,0,0
Row 5,0,0,0,0,0,0,0,0
Row 6,0,0,0,0,0,0,0,0
Row 7,0,0,0,0,0,0,-1,0


## Verification

All 56 numeric cells (8 metrics × 7 rows) are checked for agreement with the paper, with a tolerance of $\pm 1$ to account for the paper's integer rounding.

In [6]:
tol = 1
outside = diff.abs() > tol
n_total = outside.size
n_bad = int(outside.sum().sum())

print(f"{n_total - n_bad} / {n_total} cells match the paper within \u00b1{tol}.")

if n_bad:
    print("\nCells outside tolerance:")
    for case in diff.index:
        for col in metrics:
            d = int(diff.loc[case, col])
            if abs(d) > tol:
                rep = int(computed.set_index("case").loc[case, col])
                pap = int(paper.set_index("case").loc[case, col])
                print(f"  {case:7s} {col:8s}  replicated={rep:>6}  paper={pap:>6}  diff={d:+d}")

53 / 56 cells match the paper within ±1.

Cells outside tolerance:
  Row 1   Q_hat     replicated=  8485  paper=  8453  diff=+32
  Row 1   Q_star    replicated=  8485  paper=  8453  diff=+32
  Row 1   Q_0       replicated=  8485  paper=  8453  diff=+32


## Note on Row 1

Row 1 has $g/e = h/K = 1/600$, so by **Theorem 1(1)** we must have $\hat{Q} = Q^* = Q^0$. All three equal

$$\sqrt{\tfrac{2KD}{h}} \;=\; \sqrt{\tfrac{2 \cdot 180 \cdot 60{,}000}{0.3}} \;=\; \sqrt{72{,}000{,}000} \;\approx\; 8485.28.$$

The paper's own $\alpha_0 = 8485$ in the same row matches this exactly, and the row's other derived quantities ($X = -485$, $TC = 2643$, $\Delta TC = 97$) are all consistent with $Q = 8485$, not $8453$. The three `8453` entries in the $Q$ columns therefore appear to be a **typo in Table 1**; the model is reproducing the formula-correct value.